<a href="https://colab.research.google.com/github/SILAS-RANSFORD-OSEI/reliability-aware-ssdu-mri/blob/main/notebooks/05_leave_one_volume_out_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Mount Drive**

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


**Clone or pull repo**

In [2]:
import os
import subprocess

repo_url = "https://github.com/SILAS-RANSFORD-OSEI/reliability-aware-ssdu-mri.git"
repo_dir = "/content/reliability-aware-ssdu-mri"

if not os.path.exists(repo_dir):
    print("Cloning repository...")
    subprocess.run(["git", "clone", repo_url, repo_dir], check=True)
else:
    print("Repository already exists. Pulling latest changes...")
    subprocess.run(["git", "-C", repo_dir, "pull"], check=True)

os.chdir(repo_dir)
print("Current directory:", os.getcwd())

Cloning repository...
Current directory: /content/reliability-aware-ssdu-mri


**Check GPU**

In [3]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if device.type != "cuda":
    raise RuntimeError("GPU is not available. Stop here. Do not run Experiment 029 on CPU.")

Device: cuda


**Imports**

In [4]:
import sys
import os
import shutil
import h5py
import gc
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

src_path = "/content/reliability-aware-ssdu-mri/src"

if src_path not in sys.path:
    sys.path.insert(0, src_path)

from fastmri_data import load_fastmri_file
from four_way_reliability import run_four_way_reliability_experiment
from calibration import gradient_magnitude
from reliability import normalize_map, map_alignment
from models.reliability_cnn import ReliabilityCNN

print("All project modules imported successfully.")
print("CUDA available:", torch.cuda.is_available())

All project modules imported successfully.
CUDA available: True


**Reproducibility helpers**

In [5]:
def set_global_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_global_seed(0)

**Helper functions**

In [6]:
def to_norm_tensor_2d(x):
    """
    Convert a 2D numpy map to a normalized float tensor.
    """
    x_norm = normalize_map(x)
    return torch.from_numpy(x_norm).float()


def safe_torch_load(path):
    """
    torch.load wrapper compatible with different PyTorch versions.
    """
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


local_data_dir = "/content/fastmri_local"
os.makedirs(local_data_dir, exist_ok=True)


def copy_if_missing_or_too_small(src, dst, min_size_gb=0.5):
    """
    Copy a file from Drive to local Colab storage.
    Recopy if an incomplete local file exists.
    """
    if not os.path.exists(src):
        raise FileNotFoundError(f"Source file does not exist: {src}")

    needs_copy = True

    if os.path.exists(dst):
        size_gb = os.path.getsize(dst) / 1e9

        if size_gb >= min_size_gb:
            needs_copy = False
            print(f"Existing file looks valid: {dst}, size {size_gb:.3f} GB")
        else:
            print(f"Existing file too small: {size_gb:.6f} GB. Recopying...")
            os.remove(dst)

    if needs_copy:
        print(f"Copying {os.path.basename(src)}...")
        shutil.copy2(src, dst)
        size_gb = os.path.getsize(dst) / 1e9
        print(f"Copied file: {dst}, size {size_gb:.3f} GB")

    return dst


def inspect_local_h5(file_path):
    """
    Inspect metadata without loading full k-space.
    """
    with h5py.File(file_path, "r") as hf:
        print("\nFile:", os.path.basename(file_path))
        print("Keys:", list(hf.keys()))
        print("kspace shape:", hf["kspace"].shape)
        print("mask shape:", hf["mask"].shape)
        print("acquisition:", hf.attrs.get("acquisition"))
        print("acceleration:", hf.attrs.get("acceleration"))

**Define the five volumes**

In [7]:
h5_dir = "/content/drive/MyDrive/fastMRI/brain_multicoil_test/extracted/multicoil_test"

volume_paths_drive = [
    f"{h5_dir}/file_brain_AXT2_200_6002495.h5",
    f"{h5_dir}/file_brain_AXT2_200_6002623.h5",
    f"{h5_dir}/file_brain_AXT2_200_6002398.h5",
    f"{h5_dir}/file_brain_AXT2_200_2000341.h5",
    f"{h5_dir}/file_brain_AXT2_200_2000271.h5",
]

print("Checking Drive paths...\n")

for p in volume_paths_drive:
    print(os.path.exists(p), p)

Checking Drive paths...

True /content/drive/MyDrive/fastMRI/brain_multicoil_test/extracted/multicoil_test/file_brain_AXT2_200_6002495.h5
True /content/drive/MyDrive/fastMRI/brain_multicoil_test/extracted/multicoil_test/file_brain_AXT2_200_6002623.h5
True /content/drive/MyDrive/fastMRI/brain_multicoil_test/extracted/multicoil_test/file_brain_AXT2_200_6002398.h5
True /content/drive/MyDrive/fastMRI/brain_multicoil_test/extracted/multicoil_test/file_brain_AXT2_200_2000341.h5
True /content/drive/MyDrive/fastMRI/brain_multicoil_test/extracted/multicoil_test/file_brain_AXT2_200_2000271.h5


**Copy all five volumes locally**

In [8]:
volume_paths = []

for src in volume_paths_drive:
    dst = os.path.join(local_data_dir, os.path.basename(src))
    local_path = copy_if_missing_or_too_small(src, dst, min_size_gb=0.5)
    volume_paths.append(local_path)

print("\nLocal volumes:")
for p in volume_paths:
    print(p)

Copying file_brain_AXT2_200_6002495.h5...
Copied file: /content/fastmri_local/file_brain_AXT2_200_6002495.h5, size 0.623 GB
Copying file_brain_AXT2_200_6002623.h5...
Copied file: /content/fastmri_local/file_brain_AXT2_200_6002623.h5, size 0.623 GB
Copying file_brain_AXT2_200_6002398.h5...
Copied file: /content/fastmri_local/file_brain_AXT2_200_6002398.h5, size 0.623 GB
Copying file_brain_AXT2_200_2000341.h5...
Copied file: /content/fastmri_local/file_brain_AXT2_200_2000341.h5, size 0.623 GB
Copying file_brain_AXT2_200_2000271.h5...
Copied file: /content/fastmri_local/file_brain_AXT2_200_2000271.h5, size 0.623 GB

Local volumes:
/content/fastmri_local/file_brain_AXT2_200_6002495.h5
/content/fastmri_local/file_brain_AXT2_200_6002623.h5
/content/fastmri_local/file_brain_AXT2_200_6002398.h5
/content/fastmri_local/file_brain_AXT2_200_2000341.h5
/content/fastmri_local/file_brain_AXT2_200_2000271.h5


**Verify metadata**

In [9]:
for p in volume_paths:
    inspect_local_h5(p)


File: file_brain_AXT2_200_6002495.h5
Keys: ['ismrmrd_header', 'kspace', 'mask']
kspace shape: (16, 16, 768, 396)
mask shape: (396,)
acquisition: AXT2
acceleration: 4

File: file_brain_AXT2_200_6002623.h5
Keys: ['ismrmrd_header', 'kspace', 'mask']
kspace shape: (16, 16, 768, 396)
mask shape: (396,)
acquisition: AXT2
acceleration: 4

File: file_brain_AXT2_200_6002398.h5
Keys: ['ismrmrd_header', 'kspace', 'mask']
kspace shape: (16, 16, 768, 396)
mask shape: (396,)
acquisition: AXT2
acceleration: 4

File: file_brain_AXT2_200_2000341.h5
Keys: ['ismrmrd_header', 'kspace', 'mask']
kspace shape: (16, 16, 768, 396)
mask shape: (396,)
acquisition: AXT2
acceleration: 4

File: file_brain_AXT2_200_2000271.h5
Keys: ['ismrmrd_header', 'kspace', 'mask']
kspace shape: (16, 16, 768, 396)
mask shape: (396,)
acquisition: AXT2
acceleration: 4


**Define full-sample preparation function**

In [10]:
processed_dir = "/content/drive/MyDrive/fastMRI/processed_experiment_029"
os.makedirs(processed_dir, exist_ok=True)

print("Processed cache directory:", processed_dir)


def prepare_full_samples_for_volume(
    volume_path,
    volume_id,
    coil_index=0,
    train_fraction=0.2,
    cal_fraction=0.1,
    eval_fraction=0.1,
    split_seed=42,
    model_seed=0,
    num_steps=50,
    num_samples=8,
    features=16,
    dropout_p=0.1,
    lr=1e-4,
    lambda_img=1e7,
):
    print("\n===================================================")
    print("Preparing FULL samples for volume:", volume_id)
    print("Path:", volume_path)
    print("===================================================")

    data = load_fastmri_file(volume_path)

    kspace_vol = data["kspace"]
    mask_vol = data["mask"]

    print("kspace shape:", kspace_vol.shape)
    print("mask shape:", mask_vol.shape)
    print("acquisition:", data["attrs"].get("acquisition"))
    print("acceleration:", data["attrs"].get("acceleration"))

    volume_samples = []

    for slice_index in range(kspace_vol.shape[0]):
        print(f"\nPreparing {volume_id}, slice {slice_index}...")

        result = run_four_way_reliability_experiment(
            kspace=kspace_vol,
            mask=mask_vol,
            slice_index=slice_index,
            coil_index=coil_index,
            train_fraction=train_fraction,
            cal_fraction=cal_fraction,
            eval_fraction=eval_fraction,
            split_seed=split_seed,
            model_seed=model_seed,
            num_steps=num_steps,
            num_samples=num_samples,
            features=features,
            dropout_p=dropout_p,
            lr=lr,
            lambda_img=lambda_img,
        )

        x_input = result["x_input"]
        mean_image = result["mean_image"]
        uncertainty_map = result["uncertainty_map"]
        residual_cal = result["residual_energy_cal"]
        residual_eval = result["residual_energy_eval"]

        edge_map = gradient_magnitude(mean_image)

        feature_input = to_norm_tensor_2d(x_input)
        feature_recon = to_norm_tensor_2d(mean_image)
        feature_edge = to_norm_tensor_2d(edge_map)

        features_struct = torch.stack(
            [feature_input, feature_recon, feature_edge],
            dim=0
        ).unsqueeze(0)

        target_cal = to_norm_tensor_2d(residual_cal).unsqueeze(0).unsqueeze(0)

        volume_samples.append({
            "volume_id": volume_id,
            "filename": os.path.basename(volume_path),
            "slice_index": slice_index,
            "features": features_struct,
            "target_cal": target_cal,
            "residual_cal": residual_cal,
            "residual_eval": residual_eval,
            "x_input": x_input,
            "mean_image": mean_image,
            "edge_map": edge_map,
            "uncertainty_map": uncertainty_map,
        })

        del result
        gc.collect()
        torch.cuda.empty_cache()

    del data, kspace_vol, mask_vol
    gc.collect()
    torch.cuda.empty_cache()

    return volume_samples

Processed cache directory: /content/drive/MyDrive/fastMRI/processed_experiment_029


**Build or load cached full samples for all five volumes**

In [11]:
all_volume_samples = {}

for volume_idx, volume_path in enumerate(volume_paths):
    filename = os.path.basename(volume_path)
    volume_id = f"V{volume_idx+1}_{filename}"

    cache_path = os.path.join(
        processed_dir,
        f"full_samples_{filename.replace('.h5', '')}.pt"
    )

    if os.path.exists(cache_path):
        print(f"\nLoading cached full samples for {filename}")
        volume_samples = safe_torch_load(cache_path)
    else:
        volume_samples = prepare_full_samples_for_volume(
            volume_path=volume_path,
            volume_id=volume_id,
            coil_index=0,
            num_steps=50,
            num_samples=8,
            features=16,
            dropout_p=0.1,
            lr=1e-4,
            lambda_img=1e7,
        )

        torch.save(volume_samples, cache_path)
        print("Saved full sample cache:", cache_path)

    all_volume_samples[filename] = volume_samples

print("\nPrepared volumes:", list(all_volume_samples.keys()))

total_samples = sum(len(v) for v in all_volume_samples.values())
print("Total full samples:", total_samples)

first_key = list(all_volume_samples.keys())[0]
print("Example feature shape:", all_volume_samples[first_key][0]["features"].shape)
print("Example target shape:", all_volume_samples[first_key][0]["target_cal"].shape)


Preparing FULL samples for volume: V1_file_brain_AXT2_200_6002495.h5
Path: /content/fastmri_local/file_brain_AXT2_200_6002495.h5
kspace shape: (16, 16, 768, 396)
mask shape: (396,)
acquisition: AXT2
acceleration: 4

Preparing V1_file_brain_AXT2_200_6002495.h5, slice 0...

Preparing V1_file_brain_AXT2_200_6002495.h5, slice 1...

Preparing V1_file_brain_AXT2_200_6002495.h5, slice 2...

Preparing V1_file_brain_AXT2_200_6002495.h5, slice 3...

Preparing V1_file_brain_AXT2_200_6002495.h5, slice 4...

Preparing V1_file_brain_AXT2_200_6002495.h5, slice 5...

Preparing V1_file_brain_AXT2_200_6002495.h5, slice 6...

Preparing V1_file_brain_AXT2_200_6002495.h5, slice 7...

Preparing V1_file_brain_AXT2_200_6002495.h5, slice 8...

Preparing V1_file_brain_AXT2_200_6002495.h5, slice 9...

Preparing V1_file_brain_AXT2_200_6002495.h5, slice 10...

Preparing V1_file_brain_AXT2_200_6002495.h5, slice 11...

Preparing V1_file_brain_AXT2_200_6002495.h5, slice 12...

Preparing V1_file_brain_AXT2_200_600249

**Verify cached sample structure**

In [12]:
volume_names = list(all_volume_samples.keys())

print("Volumes:")
for name in volume_names:
    print(name, "num samples:", len(all_volume_samples[name]))

for name in volume_names:
    assert len(all_volume_samples[name]) == 16, f"{name} does not have 16 slices."

print("\nAll volumes verified.")

Volumes:
file_brain_AXT2_200_6002495.h5 num samples: 16
file_brain_AXT2_200_6002623.h5 num samples: 16
file_brain_AXT2_200_6002398.h5 num samples: 16
file_brain_AXT2_200_2000341.h5 num samples: 16
file_brain_AXT2_200_2000271.h5 num samples: 16

All volumes verified.


**Define fold training function**

In [13]:
def train_reliability_cnn_for_fold(
    train_samples,
    fold_id,
    num_epochs=30,
    lr=1e-3,
    in_channels=3,
    features=16,
    seed=0,
):
    set_global_seed(seed)

    model = ReliabilityCNN(
        in_channels=in_channels,
        features=features
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    epoch_losses = []

    for epoch in range(num_epochs):
        model.train()
        batch_losses = []

        perm = np.random.permutation(len(train_samples))

        for idx in perm:
            item = train_samples[idx]

            features_tensor = item["features"].to(device)
            target_tensor = item["target_cal"].to(device)

            optimizer.zero_grad()

            pred = model(features_tensor)

            pred_norm = (
                pred - pred.min()
            ) / (
                pred.max() - pred.min() + 1e-12
            )

            loss = torch.mean((pred_norm - target_tensor) ** 2)

            loss.backward()
            optimizer.step()

            batch_losses.append(loss.item())

        mean_epoch_loss = float(np.mean(batch_losses))
        epoch_losses.append(mean_epoch_loss)

        if epoch % 5 == 0 or epoch == num_epochs - 1:
            print(
                f"Fold {fold_id} | Epoch {epoch:03d} | "
                f"Mean reliability loss: {mean_epoch_loss:.6f}"
            )

        gc.collect()
        torch.cuda.empty_cache()

    return model, epoch_losses

**Define fold evaluation function**

In [14]:
def evaluate_reliability_cnn_on_volume(
    model,
    test_samples,
    fold_id,
    test_volume_name,
):
    model.eval()

    fold_results = []

    with torch.no_grad():
        for item in test_samples:
            slice_index = item["slice_index"]

            features_tensor = item["features"].to(device)

            pred = model(features_tensor)
            R_net = pred.detach().cpu().numpy()[0, 0]

            residual_cal = item["residual_cal"]
            residual_eval = item["residual_eval"]

            x_input = item["x_input"]
            mean_image = item["mean_image"]
            edge_map = item["edge_map"]
            uncertainty_map = item["uncertainty_map"]

            net_cal_alignment = map_alignment(R_net, residual_cal)
            net_eval_alignment = map_alignment(R_net, residual_eval)

            input_eval_alignment = map_alignment(x_input, residual_eval)
            mean_eval_alignment = map_alignment(mean_image, residual_eval)
            edge_eval_alignment = map_alignment(edge_map, residual_eval)
            dropout_eval_alignment = map_alignment(uncertainty_map, residual_eval)

            fold_results.append({
                "fold_id": fold_id,
                "test_volume": test_volume_name,
                "slice_index": slice_index,
                "net_cal_alignment": net_cal_alignment,
                "net_eval_alignment": net_eval_alignment,
                "input_eval_alignment": input_eval_alignment,
                "mean_eval_alignment": mean_eval_alignment,
                "edge_eval_alignment": edge_eval_alignment,
                "dropout_eval_alignment": dropout_eval_alignment,
                "net_minus_input": net_eval_alignment - input_eval_alignment,
                "net_minus_dropout": net_eval_alignment - dropout_eval_alignment,
                "net_minus_edge": net_eval_alignment - edge_eval_alignment,
            })

    return fold_results

**leave-one-volume-out validation**

In [15]:
results_dir = "/content/drive/MyDrive/fastMRI/results_experiment_029"
checkpoint_dir = "/content/drive/MyDrive/fastMRI/checkpoints_experiment_029"

os.makedirs(results_dir, exist_ok=True)
os.makedirs(checkpoint_dir, exist_ok=True)

all_fold_results = []
fold_training_logs = {}

num_epochs = 30

for fold_idx, test_volume_name in enumerate(volume_names):
    fold_id = fold_idx + 1

    print("\n===================================================")
    print(f"Starting Fold {fold_id}")
    print("Held-out test volume:", test_volume_name)
    print("===================================================")

    train_volume_names = [
        name for name in volume_names
        if name != test_volume_name
    ]

    train_samples = []
    for name in train_volume_names:
        train_samples.extend(all_volume_samples[name])

    test_samples = all_volume_samples[test_volume_name]

    print("Train volumes:", train_volume_names)
    print("Number of train samples:", len(train_samples))
    print("Number of test samples:", len(test_samples))

    assert len(train_samples) == 64
    assert len(test_samples) == 16

    model, epoch_losses = train_reliability_cnn_for_fold(
        train_samples=train_samples,
        fold_id=fold_id,
        num_epochs=num_epochs,
        lr=1e-3,
        in_channels=3,
        features=16,
        seed=fold_id,
    )

    fold_training_logs[f"fold_{fold_id}"] = {
        "test_volume": test_volume_name,
        "train_volumes": train_volume_names,
        "epoch_losses": epoch_losses,
    }

    checkpoint_path = os.path.join(
        checkpoint_dir,
        f"experiment_029_fold_{fold_id}_heldout_{test_volume_name.replace('.h5', '')}.pt"
    )

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "in_channels": 3,
            "features": 16,
            "fold_id": fold_id,
            "test_volume": test_volume_name,
            "train_volumes": train_volume_names,
            "epoch_losses": epoch_losses,
            "experiment": "029_leave_one_volume_out_validation",
        },
        checkpoint_path,
    )

    print("Saved fold checkpoint:", checkpoint_path)

    fold_results = evaluate_reliability_cnn_on_volume(
        model=model,
        test_samples=test_samples,
        fold_id=fold_id,
        test_volume_name=test_volume_name,
    )

    all_fold_results.extend(fold_results)

    fold_df = pd.DataFrame(fold_results)

    fold_csv_path = os.path.join(
        results_dir,
        f"experiment_029_fold_{fold_id}_slice_results.csv"
    )

    fold_df.to_csv(fold_csv_path, index=False)
    print("Saved fold results:", fold_csv_path)

    del model
    gc.collect()
    torch.cuda.empty_cache()

print("\nAll folds complete.")


Starting Fold 1
Held-out test volume: file_brain_AXT2_200_6002495.h5
Train volumes: ['file_brain_AXT2_200_6002623.h5', 'file_brain_AXT2_200_6002398.h5', 'file_brain_AXT2_200_2000341.h5', 'file_brain_AXT2_200_2000271.h5']
Number of train samples: 64
Number of test samples: 16
Fold 1 | Epoch 000 | Mean reliability loss: 0.013606
Fold 1 | Epoch 005 | Mean reliability loss: 0.003180
Fold 1 | Epoch 010 | Mean reliability loss: 0.003057
Fold 1 | Epoch 015 | Mean reliability loss: 0.003008
Fold 1 | Epoch 020 | Mean reliability loss: 0.002983
Fold 1 | Epoch 025 | Mean reliability loss: 0.002944
Fold 1 | Epoch 029 | Mean reliability loss: 0.002924
Saved fold checkpoint: /content/drive/MyDrive/fastMRI/checkpoints_experiment_029/experiment_029_fold_1_heldout_file_brain_AXT2_200_6002495.pt
Saved fold results: /content/drive/MyDrive/fastMRI/results_experiment_029/experiment_029_fold_1_slice_results.csv

Starting Fold 2
Held-out test volume: file_brain_AXT2_200_6002623.h5
Train volumes: ['file_brai

**Create full slice-level results table**

In [16]:
experiment_029_df = pd.DataFrame(all_fold_results)

print("Total rows:", len(experiment_029_df))
experiment_029_df.head()

Total rows: 80


,fold_id,test_volume,slice_index,net_cal_alignment,net_eval_alignment,input_eval_alignment,mean_eval_alignment,edge_eval_alignment,dropout_eval_alignment,net_minus_input,net_minus_dropout,net_minus_edge
0,1,file_brain_AXT2_200_6002495.h5,0,0.484335,0.499505,0.478424,0.415558,0.303639,0.381174,0.021081,0.118331,0.195866
1,1,file_brain_AXT2_200_6002495.h5,1,0.476983,0.526753,0.491943,0.433852,0.319824,0.413082,0.034810,0.113671,0.206929
2,1,file_brain_AXT2_200_6002495.h5,2,0.465786,0.551470,0.536322,0.431318,0.368996,0.438849,0.015148,0.112621,0.182474
3,1,file_brain_AXT2_200_6002495.h5,3,0.448367,0.561810,0.554698,0.450208,0.370511,0.453341,0.007112,0.108469,0.191299
4,1,file_brain_AXT2_200_6002495.h5,4,0.482719,0.496058,0.495757,0.414697,0.334013,0.404839,0.000301,0.091219,0.162045


In [17]:
# SAVE
all_results_path = os.path.join(
    results_dir,
    "experiment_029_all_slice_results.csv"
)

experiment_029_df.to_csv(all_results_path, index=False)

print("Saved all slice results:", all_results_path)

Saved all slice results: /content/drive/MyDrive/fastMRI/results_experiment_029/experiment_029_all_slice_results.csv


**Volume-level summary**

In [18]:
experiment_029_volume_summary = (
    experiment_029_df
    .groupby(["fold_id", "test_volume"])
    .agg(
        net_mean=("net_eval_alignment", "mean"),
        net_std=("net_eval_alignment", "std"),
        input_mean=("input_eval_alignment", "mean"),
        input_std=("input_eval_alignment", "std"),
        dropout_mean=("dropout_eval_alignment", "mean"),
        edge_mean=("edge_eval_alignment", "mean"),
        mean_recon_mean=("mean_eval_alignment", "mean"),
        margin_input_mean=("net_minus_input", "mean"),
        margin_input_std=("net_minus_input", "std"),
        margin_dropout_mean=("net_minus_dropout", "mean"),
        margin_edge_mean=("net_minus_edge", "mean"),
    )
    .reset_index()
)

experiment_029_volume_summary

,fold_id,test_volume,net_mean,net_std,input_mean,input_std,dropout_mean,edge_mean,mean_recon_mean,margin_input_mean,margin_input_std,margin_dropout_mean,margin_edge_mean
0,1,file_brain_AXT2_200_6002495.h5,0.565702,0.044425,0.552900,0.040571,0.470354,0.394552,0.459942,0.012802,0.015936,0.095348,0.171150
1,2,file_brain_AXT2_200_6002623.h5,0.573267,0.037467,0.554591,0.023771,0.515027,0.424764,0.555256,0.018675,0.031687,0.058239,0.148503
2,3,file_brain_AXT2_200_6002398.h5,0.544925,0.055589,0.536431,0.078540,0.444946,0.344193,0.495097,0.008494,0.032068,0.099978,0.200731
3,4,file_brain_AXT2_200_2000341.h5,0.591613,0.032175,0.564113,0.036672,0.495161,0.423609,0.483928,0.027500,0.029228,0.096452,0.168004
4,5,file_brain_AXT2_200_2000271.h5,0.580816,0.052049,0.582079,0.045057,0.524514,0.415198,0.495975,-0.001263,0.015743,0.056302,0.165618
